<a href="https://colab.research.google.com/github/sswess122-star/full-outer-join/blob/main/G6_DV_L13_Window_Functions_SQLite3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd

print("pandas:", pd.__version__)
print("SQLite:", sqlite3.sqlite_version)

conn = sqlite3.connect(":memory:")
print("SQLite database connected.")

pandas: 2.2.3
SQLite: 3.45.1
SQLite database connected.


In [2]:
conn.executescript('''
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    track TEXT NOT NULL,
    math_score INTEGER NOT NULL,
    science_score INTEGER NOT NULL,
    project_score INTEGER NOT NULL
);

CREATE TABLE enrolments (
    enrolment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_name TEXT NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
);

CREATE TABLE assessments (
    assessment_id INTEGER PRIMARY KEY,
    enrolment_id INTEGER NOT NULL,
    assessment_date TEXT NOT NULL,
    score REAL NOT NULL,
    FOREIGN KEY (enrolment_id) REFERENCES enrolments(enrolment_id)
);
''')

students = [
    (1, "Ada", "Dataville", 92, 88, 90),
    (2, "Ben", "Dataville", 85, 82, 86),
    (3, "Chika", "Dataville", 85, 79, 88),
    (4, "David", "Dataville", 72, 75, 78),
    (5, "Efe", "Mindspire", 95, 91, 94),
    (6, "Favour", "Mindspire", 88, 84, 89),
    (7, "Grace", "Mindspire", 88, 86, 87),
    (8, "Hassan", "Mindspire", 70, 73, 76),
    (9, "Ivy", "Webweave", 91, 89, 93),
    (10, "John", "Webweave", 76, 80, 82)
]

conn.executemany(
    '''INSERT INTO students
       (student_id, name, track, math_score, science_score, project_score)
       VALUES (?, ?, ?, ?, ?, ?)''',
    students
)

enrolments = [
    (1,1,"Python Foundations"), (2,2,"Python Foundations"),
    (3,3,"Python Foundations"), (4,4,"Python Foundations"),
    (5,5,"Data Analytics"), (6,6,"Data Analytics"),
    (7,7,"Data Analytics"), (8,8,"Data Analytics"),
    (9,9,"Web Development"), (10,10,"Web Development")
]

conn.executemany(
    '''INSERT INTO enrolments
       (enrolment_id, student_id, course_name)
       VALUES (?, ?, ?)''',
    enrolments
)

assessments = [
    (1,1,"2026-01-10",68),(2,1,"2026-02-10",75),(3,1,"2026-03-10",84),
    (4,2,"2026-01-10",72),(5,2,"2026-02-10",70),(6,2,"2026-03-10",78),
    (7,3,"2026-01-10",80),(8,3,"2026-02-10",84),(9,3,"2026-03-10",86),
    (10,4,"2026-01-10",60),(11,4,"2026-02-10",64),(12,4,"2026-03-10",62),
    (13,5,"2026-01-10",78),(14,5,"2026-02-10",86),(15,5,"2026-03-10",94),
    (16,6,"2026-01-10",70),(17,6,"2026-02-10",76),(18,6,"2026-03-10",81),
    (19,7,"2026-01-10",82),(20,7,"2026-02-10",80),(21,7,"2026-03-10",88),
    (22,8,"2026-01-10",65),(23,8,"2026-02-10",68),(24,8,"2026-03-10",66),
    (25,9,"2026-01-10",75),(26,9,"2026-02-10",85),(27,9,"2026-03-10",91),
    (28,10,"2026-01-10",69),(29,10,"2026-02-10",74),(30,10,"2026-03-10",79)
]

conn.executemany(
    '''INSERT INTO assessments
       (assessment_id, enrolment_id, assessment_date, score)
       VALUES (?, ?, ?, ?)''',
    assessments
)

conn.commit()

students_df = pd.read_sql_query("SELECT * FROM students", conn)
enrolments_df = pd.read_sql_query("SELECT * FROM enrolments", conn)
assessments_df = pd.read_sql_query("SELECT * FROM assessments", conn)

display(students_df)

,student_id,name,track,math_score,science_score,project_score
0,1,Ada,Dataville,92,88,90
1,2,Ben,Dataville,85,82,86
2,3,Chika,Dataville,85,79,88
3,4,David,Dataville,72,75,78
4,5,Efe,Mindspire,95,91,94
5,6,Favour,Mindspire,88,84,89
6,7,Grace,Mindspire,88,86,87
7,8,Hassan,Mindspire,70,73,76
8,9,Ivy,Webweave,91,89,93
9,10,John,Webweave,76,80,82


In [ ]:
sql = '''
SELECT
    name,
    track,
    math_score,
    ROW_NUMBER() OVER (
        ORDER BY math_score DESC, name
    ) AS global_number
FROM students
ORDER BY global_number;
'''

df_row_number = pd.read_sql_query(sql, conn)
display(df_row_number)

,name,track,math_score,global_number
0,Efe,Mindspire,95,1
1,Ada,Dataville,92,2
2,Ivy,Webweave,91,3
3,Favour,Mindspire,88,4
4,Grace,Mindspire,88,5
5,Ben,Dataville,85,6
6,Chika,Dataville,85,7
7,John,Webweave,76,8
8,David,Dataville,72,9
9,Hassan,Mindspire,70,10


In [ ]:
sql = '''
SELECT
    name,
    track,
    math_score,
    RANK() OVER (
        PARTITION BY track
        ORDER BY math_score DESC
    ) AS track_rank
FROM students
ORDER BY track, track_rank, name;
'''

df_track_rank = pd.read_sql_query(sql, conn)
display(df_track_rank)

,name,track,math_score,track_rank
0,Ada,Dataville,92,1
1,Ben,Dataville,85,2
2,Chika,Dataville,85,2
3,David,Dataville,72,4
4,Efe,Mindspire,95,1
5,Favour,Mindspire,88,2
6,Grace,Mindspire,88,2
7,Hassan,Mindspire,70,4
8,Ivy,Webweave,91,1
9,John,Webweave,76,2


In [ ]:
sql = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LAG(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS previous_score
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''

df_lag = pd.read_sql_query(sql, conn)
display(df_lag)

,name,assessment_date,score,previous_score
0,Ada,2026-01-10,68.0,NaN
1,Ada,2026-02-10,75.0,68.0
2,Ada,2026-03-10,84.0,75.0
3,Ben,2026-01-10,72.0,NaN
4,Ben,2026-02-10,70.0,72.0
5,Ben,2026-03-10,78.0,70.0
6,Chika,2026-01-10,80.0,NaN
7,Chika,2026-02-10,84.0,80.0
8,Chika,2026-03-10,86.0,84.0
9,David,2026-01-10,60.0,NaN


In [ ]:
sql = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LAG(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS previous_score
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''

df_lag = pd.read_sql_query(sql, conn)
display(df_lag)

,name,assessment_date,score,previous_score
0,Ada,2026-01-10,68.0,NaN
1,Ada,2026-02-10,75.0,68.0
2,Ada,2026-03-10,84.0,75.0
3,Ben,2026-01-10,72.0,NaN
4,Ben,2026-02-10,70.0,72.0
5,Ben,2026-03-10,78.0,70.0
6,Chika,2026-01-10,80.0,NaN
7,Chika,2026-02-10,84.0,80.0
8,Chika,2026-03-10,86.0,84.0
9,David,2026-01-10,60.0,NaN


In [ ]:
sql = '''
WITH scored_students AS (
    SELECT
        name,
        track,
        math_score + science_score + project_score AS total_score
    FROM students
),
ranked_students AS (
    SELECT
        name,
        track,
        total_score,
        RANK() OVER (
            PARTITION BY track
            ORDER BY total_score DESC
        ) AS track_rank
    FROM scored_students
)
SELECT
    name,
    track,
    total_score,
    track_rank
FROM ranked_students
WHERE track_rank <= 3
ORDER BY track, track_rank, name;
'''

df_top3_total = pd.read_sql_query(sql, conn)
display(df_top3_total)

,name,track,total_score,track_rank
0,Ada,Dataville,270,1
1,Ben,Dataville,253,2
2,Chika,Dataville,252,3
3,Efe,Mindspire,280,1
4,Favour,Mindspire,261,2
5,Grace,Mindspire,261,2
6,Ivy,Webweave,273,1
7,John,Webweave,238,2


# Homework

## HW 1 — Notebook Task

Write two window-function queries:

1. For each track, find the student ranked **3rd by total score**.
2. Find students whose **most recent assessment score is lower than their previous assessment**, using `LAG()`.

For each:
- write the business question in Markdown;
- write the SQL;
- load the result into a pandas DataFrame;
- write one sentence explaining the result.

## HW 2 — Written Interpretation

Explain `RANK()` versus `DENSE_RANK()` using:

**95, 90, 90, 80, 75**

Show both outputs and explain the difference.